# 0 Imports

In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from src.suporte import funcoes_suporte as fs

## 0.1 Funções Suporte

In [3]:
fs.jupyter_settings(altura = 10, largura = 12, fonte = 8)
fs.supressao_notacao(casa_decimal = 2)

# 0.2 Load Data

In [4]:
df_all = fs.load_pickle("../data/interim/2.all_quantity.pkl")
df_all.sample(5)

,invoice_no,stock_code,quantity,invoice_date,unit_price,customer_id,country
328923,565839,22771,12,2011-09-07,1.25,13186,United Kingdom
418545,572732,16161P,25,2011-10-25,0.42,18079,United Kingdom
126822,547102,22423,1,2011-03-21,12.75,14245,United Kingdom
537450,581266,23239,6,2011-12-08,1.65,12621,Germany
420392,572892,23328,48,2011-10-26,3.39,17017,United Kingdom


In [5]:
df_compras = fs.load_pickle("../data/interim/2.df_compras.pkl")
df_compras.sample(5)

,invoice_no,stock_code,quantity,invoice_date,unit_price,customer_id,country
411240,572210,22468,2,2011-10-21,6.75,13755,United Kingdom
64563,541649,37446,8,2011-01-20,1.45,17418,United Kingdom
444318,574725,23493,3,2011-11-06,1.95,15883,United Kingdom
265114,560209,85176,2,2011-07-15,0.85,15004,United Kingdom
10097,537225,20771,2,2010-12-05,2.55,12748,United Kingdom


In [6]:
df_compras.columns

Index(['invoice_no', 'stock_code', 'quantity', 'invoice_date', 'unit_price',
       'customer_id', 'country'],
      dtype='object')

In [7]:
df_returns = fs.load_pickle("../data/interim/2.df_returns.pkl")
df_returns.sample(5)

,invoice_no,stock_code,quantity,invoice_date,unit_price,customer_id,country
226321,C556787,20914,-8,2011-06-14,2.95,16745,United Kingdom
174964,C551867,22666,-2,2011-05-04,2.95,16227,United Kingdom
422619,C573099,22654,-2,2011-10-27,5.95,12921,United Kingdom
384034,C570099,21928,-1,2011-10-07,1.65,13798,United Kingdom
44153,C540156,21843,-1,2011-01-05,10.95,12683,France


# 1.0 F.E.

In [8]:
df_ref = df_compras.drop(columns = ['invoice_no', 'stock_code', 'quantity', 'invoice_date', 'unit_price', 'country']).drop_duplicates( ignore_index=True)
df_ref.head()

,customer_id
0,17850
1,13047
2,12583
3,13748
4,15100


## 1.1 Gross Revenue (Faturamento) quantidade * preço

In [9]:
df_compras["faturamento"] = df_compras["quantity"] * df_compras["unit_price"]

## 1.2 Monetário

In [ ]:
df_monetario = df_compras[["customer_id", "faturamento"]].groupby("customer_id").sum().reset_index()

df_ref = pd.merge(df_ref, df_monetario, how="left", on="customer_id")

df_ref.head()

,customer_id,faturamento
0,17850,5391.21
1,13047,3232.59
2,12583,6705.38
3,13748,948.25
4,15100,876.00


In [11]:
df_ref.isna().sum()

customer_id    0
faturamento    0
dtype: int64

In [12]:
del df_monetario

## 1.3 Recência

In [13]:
df_recencia = df_compras[["customer_id", "invoice_date"]].groupby("customer_id").max().reset_index()

df_recencia["recencia_days"] = (df_compras["invoice_date"].max() - df_recencia["invoice_date"]).dt.days

df_recencia = df_recencia[["customer_id", "recencia_days"]].copy()

df_ref = pd.merge(df_ref, df_recencia, how = "left", on="customer_id")

df_ref.head()

,customer_id,faturamento,recencia_days
0,17850,5391.21,372
1,13047,3232.59,56
2,12583,6705.38,2
3,13748,948.25,95
4,15100,876.00,333


In [14]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
dtype: int64

In [15]:
del df_recencia

## 1.4 Frequência

In [16]:
df_frequencia = df_compras[["customer_id", "invoice_no"]].drop_duplicates().groupby("customer_id").count().reset_index().rename(columns = {"invoice_no": "frequencia"})

df_ref = pd.merge(df_ref, df_frequencia, how='left', on='customer_id')

df_ref.head()

,customer_id,faturamento,recencia_days,frequencia
0,17850,5391.21,372,34
1,13047,3232.59,56,9
2,12583,6705.38,2,15
3,13748,948.25,95,5
4,15100,876.00,333,3


In [17]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
frequencia       0
dtype: int64

In [18]:
del df_frequencia

## 1.4 Avg Ticket

In [19]:
avg_ticket = df_compras[['customer_id','faturamento']].groupby('customer_id').mean().reset_index().rename(columns = {'faturamento':'avg_faturamento'})
df_ref = pd.merge(df_ref, avg_ticket, how='left', on='customer_id')
df_ref.head()

,customer_id,faturamento,recencia_days,frequencia,avg_faturamento
0,17850,5391.21,372,34,18.15
1,13047,3232.59,56,9,18.90
2,12583,6705.38,2,15,28.90
3,13748,948.25,95,5,33.87
4,15100,876.00,333,3,292.00


In [20]:
df_ref.isna().sum()

customer_id        0
faturamento        0
recencia_days      0
frequencia         0
avg_faturamento    0
dtype: int64

## 1.5. Returns

In [21]:
df_ret = df_returns[['customer_id', 'invoice_no']].drop_duplicates().groupby('customer_id').count().reset_index().rename(columns ={'invoice_no': 'retornos'})
df_ref = pd.merge(df_ref, df_ret, how='left', on='customer_id')
df_ref.loc[df_ref['retornos'].isna(),'retornos'] = 0
df_ref.head()

,customer_id,faturamento,recencia_days,frequencia,avg_faturamento,retornos
0,17850,5391.21,372,34,18.15,1.00
1,13047,3232.59,56,9,18.90,7.00
2,12583,6705.38,2,15,28.90,2.00
3,13748,948.25,95,5,33.87,0.00
4,15100,876.00,333,3,292.00,3.00


In [22]:
df_ref.isna().sum()

customer_id        0
faturamento        0
recencia_days      0
frequencia         0
avg_faturamento    0
retornos           0
dtype: int64

# 2.0 Exportar DF

In [24]:
path = "../data/interim/3.fe.pkl"
fs.save_pickle(obj=df_ref,path=path)